# Libaries en Functies

Deze notebook wordt gebruikt om de functies te definieren voor het refresh mechanisme dat via Semantic link werkt. Dit Notebook wordt binnen dezelfde sessie aangeroepen door de refresh scripts. Wijzigingen in dit script hebben invloed op alle refresh scipts op ACC en PROD 

In [ ]:
#libaries ophalen
import pyspark.sql.functions as F
from pyspark.sql.functions import *
import pandas as pd
import sempy.fabric as fabric
import time
import datetime

In [ ]:
#functie definitie

# Vraag de status op van de refresh
def request_status(model, refresh_request_id):    
    statussen = fabric.list_refresh_requests(dataset= model, workspace= WorkspaceId)
    status = statussen[statussen['Request Id'] == refresh_request_id]['Status'].values
    if len(status) > 0:
        return status[0]
    return None

def wait_for_status_update(model, request_id):
    status = request_status(model, request_id)
    while status == "Unknown":
        time.sleep(20)  # Wacht 20 seconden
        status = request_status(model, request_id)
    return status

# functie om de tabellen te refreshen in serie. de input hiervoor is een dataframe met met partities en tabellen die je kan ophalen
# met het volgende code block
def refresh_pbi_model_1(tables_to_refresh):
    #uitvoeren van de refresh

    tables = tables_to_refresh

    for partitie in tables_to_refresh['Partition Name']:
        
        #tabel naam ophalen
        tabel = tables[tables['Partition Name'] == partitie]
        tabel_naam = str(tabel['Table Name'].values[0])

        # object om te verversen klaarzetten
        my_objects = [
            {"table": tabel_naam, "partition": partitie}  
        ]

        # ververs partitie
        request_id = fabric.refresh_dataset(dataset = model, refresh_type = 'full', objects = my_objects, workspace= WorkspaceId)

        #haal de status op

        final_status = wait_for_status_update(model, request_id)

        if final_status != "Unknown":
            continue
            


# paralel verversen

def refresh_pbi_model_2(partitieGroups_to_refresh, tables):

    for partitieGroep in partitieGroups_to_refresh:
        
        Filtered_tables = tables[tables['partitieGroep'] == partitieGroep ]
        my_objects = [
            {"table": row["Table Name"], "partition": row["Partition Name"]}
            for index, row in Filtered_tables.iterrows()
        ]
        
        request_id = fabric.refresh_dataset(
            dataset= model, 
            refresh_type= refresh_type, 
            objects= my_objects, 
            workspace= WorkspaceId, 
            max_parallelism= max_parallelism,
            retry_count = retry_count, 
            commit_mode = commit_mode
            )

        final_status = wait_for_status_update(model, request_id)

        if final_status != "Unknown":
            continue